In [ ]:
from google.colab import drive
drive.mount('/content/drive')


# C2 2A: fixed single-versus-four-block diagnostic

This entry reuses one saved FP32 terminal and compares one original central 8x8 block with four fixed, non-overlapping 8x8 blocks. It is a controlled spatial-repetition diagnostic with independent arms, not blind public calibration or a 2B experiment. It performs no Wan terminal generation or transformer loading.

In [ ]:
from pathlib import Path
import sys, subprocess
REPOSITORY_URL = 'https://github.com/RICHAAARC/SC-SSTW.git'
SOURCE_BRANCH = 'c2a-2a-colab-preparation'
SOURCE = Path('/content/c2a_multiblock_diagnostic_source')
if SOURCE.exists():
    raise FileExistsError('Use a fresh runtime; preserve existing source')
subprocess.run(['git', 'init', str(SOURCE)], check=True)
subprocess.run(['git', '-C', str(SOURCE), 'remote', 'add', 'origin', REPOSITORY_URL], check=True)
subprocess.run(['git', '-C', str(SOURCE), 'fetch', '--depth', '1', 'origin', SOURCE_BRANCH], check=True)
subprocess.run(['git', '-C', str(SOURCE), 'checkout', '--detach', 'FETCH_HEAD'], check=True)
print('Source:', subprocess.check_output(['git', '-C', str(SOURCE), 'rev-parse', 'HEAD'], text=True).strip())


In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', 'diffusers', 'transformers', 'accelerate', 'ftfy', 'sentencepiece', 'safetensors', 'huggingface_hub', 'numpy', 'Pillow'], check=True)
subprocess.run(['ffmpeg', '-version'], check=True)
# Keep Colab CUDA PyTorch; actual versions are recorded by the result.


## Fixed geometry and workload

The terminal is exactly `shared_terminal_normalized.pt` from `C2A_SecondAxis_Diagnostic/c2a_second_axis_20260914T145106Z`, shape [1,16,13,40,64], CPU float32. The single layout is top-left (16,28). The four-block layout is (16,28), (4,16), (4,40), (28,28), all non-overlapping in 40x64. Each block independently writes groups 1/2/3 with channels 0/1, beta .25 and rho .5; group 2 reads q. Outputs are shared OFF, then ZERO/+E1/-E1/+E2/-E2 for each layout: 11 outputs, 11 VAE decodes, 11 VAE encodes, 0 generation and 0 transformer forwards.

In [ ]:
from datetime import datetime, timezone
CONFIG = SOURCE / 'runtime/c2a/c2a_multiblock_diagnostic_run.json'
RUN_ID = 'c2a_multiblock_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
OUTPUT = Path('/content/drive/MyDrive/Video-WM/C2A_Multiblock_Diagnostic') / RUN_ID
print(CONFIG.read_text())
print('Output:', OUTPUT)
if OUTPUT.exists():
    raise FileExistsError(str(OUTPUT))


In [ ]:
import os, signal
command = [sys.executable, '-m', 'runtime.c2a.run_multiblock_diagnostic', '--config', str(CONFIG), '--output', str(OUTPUT), '--execute']
process = subprocess.Popen(command, cwd=SOURCE, start_new_session=True)
try:
    returncode = process.wait()
except BaseException:
    try: process.send_signal(signal.SIGTERM)
    except ProcessLookupError: pass
    try: process.wait(timeout=5)
    except subprocess.TimeoutExpired:
        try: os.killpg(process.pid, signal.SIGKILL)
        except ProcessLookupError: pass
        process.wait()
    raise
print('launcher exit', returncode)
print((OUTPUT / 'result.json').read_text() if (OUTPUT / 'result.json').exists() else 'No result file')
if returncode: raise subprocess.CalledProcessError(returncode, command)


## Persisted diagnostic package

The enabled run writes only to `MyDrive/Video-WM/C2A_Multiblock_Diagnostic/<UTC-run-id>/` and never overwrites the saved terminal source. Every readable output preserves MP4, pre-codec float RGB (about 11×92 MiB), full post-MP4 reencoded normalized latent, and necessary block records. Per-block A/b is fitted only from ZERO/+E1/+E2. For a state read, it computes each block's inverse-calibrated state first and then a fixed equal 1/4 aggregate; it does not average q then apply one block's inverse. Negative axes are held out and no training-point fit is a success claim. Any failed or inverse-unsupported block makes the four-block layout unsupported without omission, reweighting or pseudoinverse.